# 11. Scenario Report

## Objective

This notebook demonstrates scenario analysis for an Interest Rate Swap.

Scenario analysis evaluates how the value of a trade changes under different market conditions.

Workflow

Market Quotes
→ Bootstrap Yield Curve
→ Build Interest Rate Swap
→ Define Market Scenarios
→ Run Scenario Report
→ Compare Present Value and P&L

Unlike DV01, which measures sensitivity to a one basis point movement, scenario analysis allows arbitrary market shocks and forms the foundation for stress testing and Historical Value at Risk (VaR).

In [1]:
from datetime import date

from src.common.enums import (
    DayCountConvention,
    FloatingIndex,
    PayReceive,
    PaymentFrequency,
)

from src.curves.bootstrapper import Bootstrapper

from src.market_data.market_instrument import MarketInstrument
from src.market_data.market_quote_collection import MarketQuoteCollection

from src.products.trade import Trade
from src.products.fixed_leg import FixedLeg
from src.products.floating_leg import FloatingLeg
from src.products.interest_rate_swap import InterestRateSwap

from src.risk.scenario import Scenario
from src.risk.scenario_report_engine import ScenarioReportEngine

## Build Market Curve

In [2]:
quotes = MarketQuoteCollection()

quotes.add(MarketInstrument("Deposit", "1M", 0.05))
quotes.add(MarketInstrument("Deposit", "3M", 0.05))
quotes.add(MarketInstrument("Deposit", "6M", 0.05))
quotes.add(MarketInstrument("Deposit", "1Y", 0.05))
quotes.add(MarketInstrument("Deposit", "2Y", 0.05))
quotes.add(MarketInstrument("Deposit", "3Y", 0.05))

curve = Bootstrapper(
    valuation_date=date(2026, 1, 1),
    market_quotes=quotes,
).build()

curve

YieldCurve(valuation_date=2026-01-01, points=6)

## Build Interest Rate Swap

In [3]:
trade = Trade(
    trade_id="IRS001",
    counterparty="ABC",
    currency="USD",
    effective_date=date(2026, 1, 1),
    maturity_date=date(2029, 1, 1),
)

fixed_leg = FixedLeg(
    notional=1_000_000,
    fixed_rate=0.05,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.RECEIVE,
)

floating_leg = FloatingLeg(
    notional=1_000_000,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.PAY,
    floating_index=FloatingIndex.SOFR,
)

swap = InterestRateSwap(
    trade=trade,
    fixed_leg=fixed_leg,
    floating_leg=floating_leg,
)

swap

InterestRateSwap(trade=Trade(trade_id='IRS001', counterparty='ABC', currency='USD', effective_date=datetime.date(2026, 1, 1), maturity_date=datetime.date(2029, 1, 1)), fixed_leg=FixedLeg(notional=1000000, payment_frequency=<PaymentFrequency.ANNUAL: 1>, day_count=<DayCountConvention.ACT_365: 'ACT/365'>, pay_receive=<PayReceive.RECEIVE: 'RECEIVE'>, fixed_rate=0.05), floating_leg=FloatingLeg(notional=1000000, payment_frequency=<PaymentFrequency.ANNUAL: 1>, day_count=<DayCountConvention.ACT_365: 'ACT/365'>, pay_receive=<PayReceive.PAY: 'PAY'>, floating_index=<FloatingIndex.SOFR: 'SOFR'>, spread=0.0))

## Define Market Scenarios

In [5]:
scenarios = [

    Scenario(
        name="Base",
        parallel_shift=0.0,
    ),

    Scenario(
        name="Parallel Up 10bp",
        parallel_shift=0.001,
    ),

    Scenario(
        name="Parallel Down 10bp",
        parallel_shift=-0.001,
    ),
]

## Run Scenario Report

In [6]:
report = ScenarioReportEngine().run(
    swap,
    curve,
    scenarios,
)

In [7]:
print(
    f"{'Scenario':<25}"
    f"{'Present Value':>20}"
    f"{'PnL':>15}"
)

print("-" * 60)

for result in report.results:

    print(
        f"{result.scenario:<25}"
        f"{result.present_value:>20,.2f}"
        f"{result.pnl:>15,.2f}"
    )

Scenario                        Present Value            PnL
------------------------------------------------------------
Base                               135,956.65           0.00
Parallel Up 10bp                   135,689.45        -267.20
Parallel Down 10bp                 136,224.46         267.81


## Interpretation

The report compares the swap's present value under multiple market scenarios.

- **Base** represents the current market.
- **Parallel Up** shifts the entire yield curve upward by 10 basis points.
- **Parallel Down** shifts the entire curve downward by 10 basis points.

The difference between the stressed present value and the base present value is the scenario P&L.

Scenario analysis is widely used for:

- Market Risk Monitoring
- Stress Testing
- Regulatory Capital Analysis
- Historical Value at Risk (VaR)

In the next notebook, we will extend this framework to Historical VaR by replacing manually defined scenarios with historical market movements.